# Clean foundation

## Filtering rules
1. Drop invoices starting with `C`.
2. Keep rows where `Quantity > 0` and `UnitPrice > 0`.
3. Keep product-like stock codes matching five digits plus optional letters.
4. Match cancellation lines to one earlier positive line with the same customer, stock code, and exact quantity; remove only matched positive lines. Each positive line can be matched once, choosing the closest earlier line.
5. Keep rows with missing `CustomerID`.

Added columns: `line_value = Quantity x UnitPrice` (recorded value, not verified revenue) and `week` (the start date of the invoice week).

Still uncertain: unmatched, partial, or altered-quantity cancellations cannot be linked by this exact rule; returns may refer to sales before the file starts; zero-price rows and nonstandard product codes are excluded by the stated rules.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

project_root_candidates_03 = (Path.cwd(), *Path.cwd().parents)
project_root_03 = next(
    (candidate for candidate in project_root_candidates_03 if (candidate / "src" / "clean.py").is_file()),
    None,
)
if project_root_03 is None:
    raise FileNotFoundError("Could not locate the project root from the current directory.")
if str(project_root_03) not in sys.path:
    sys.path.insert(0, str(project_root_03))
from src.clean import PRODUCT_CODE, load_clean, match_cancelled_orders

relative_data_path_03 = Path("data") / "raw" / "Online Retail.xlsx"
data_path_03 = project_root_03 / relative_data_path_03
print(f"Workbook: {relative_data_path_03}")
raw_03 = pd.read_excel(data_path_03, engine="openpyxl")
raw_03["StockCode"] = raw_03["StockCode"].astype(str).str.strip()
matched_positive_indices_03, matched_cancellation_indices_03 = match_cancelled_orders(raw_03)
clean_03 = load_clean(path=data_path_03, verbose=True)

assert clean_03["Quantity"].gt(0).all()
assert clean_03["UnitPrice"].gt(0).all()
assert not clean_03["InvoiceNo"].astype(str).str.startswith("C").any()
assert clean_03["StockCode"].str.match(PRODUCT_CODE).all()
print("Clean-data filter assertions passed.")

Workbook: data\raw\Online Retail.xlsx


Matched cancellations: 2,895
Rows removed by PRODUCT_CODE regex (top 20 StockCodes):


StockCode
POST            1256
DOT              710
M                571
C2               144
D                 77
S                 63
BANK CHARGES      37
AMAZONFEE         34
CRUK              16
DCGSSGIRL         13
DCGSSBOY          11
gift_0001_20      10
gift_0001_10       9
gift_0001_30       8
DCGS0003           5
gift_0001_50       4
PADS               4
gift_0001_40       3
B                  3
DCGS0076           2
541,909 raw rows -> 524,894 clean rows, 3,895 products, 5,351,288 units


Clean-data filter assertions passed.


## Exact cancellation matching summary

A cancellation is considered only when its invoice starts with `C`, quantity is negative, unit price is positive, and customer ID is present. Matching also requires an earlier positive line with the same customer, stock code, and absolute quantity.

In [2]:
considered_cancellations_03 = (
    raw_03["InvoiceNo"].astype(str).str.startswith("C")
    & raw_03["Quantity"].lt(0)
    & raw_03["UnitPrice"].gt(0)
    & raw_03["CustomerID"].notna()
)
matched_pairs_03 = pd.DataFrame({
    "InvoiceNo": raw_03.loc[matched_positive_indices_03, "InvoiceNo"].to_numpy(),
    "C-InvoiceNo": raw_03.loc[matched_cancellation_indices_03, "InvoiceNo"].to_numpy(),
    "StockCode": raw_03.loc[matched_positive_indices_03, "StockCode"].to_numpy(),
    "Quantity": raw_03.loc[matched_positive_indices_03, "Quantity"].to_numpy(),
    "CustomerID": raw_03.loc[matched_positive_indices_03, "CustomerID"].to_numpy(),
})
matched_quantities_03 = matched_pairs_03["Quantity"]
print(f"Cancellation lines considered: {int(considered_cancellations_03.sum()):,}")
print(f"Cancellation lines matched: {len(matched_cancellation_indices_03):,}")
print(f"Cancellation lines unmatched: {int(considered_cancellations_03.sum()) - len(matched_cancellation_indices_03):,}")
print(f"Positive lines excluded: {len(matched_positive_indices_03):,}")
print(f"Units in positive lines excluded: {int(raw_03.loc[matched_positive_indices_03, 'Quantity'].sum()):,}")
print("10 largest matched pairs:")
display(
    matched_pairs_03.sort_values(
        ["Quantity", "InvoiceNo"], ascending=[False, True], kind="mergesort"
    ).head(10).reset_index(drop=True)
)
print(
    "Matched quantity distribution: "
    f"min={matched_quantities_03.min():,}, "
    f"median={matched_quantities_03.median():g}, "
    f"90th percentile={matched_quantities_03.quantile(0.9):g}, "
    f"max={matched_quantities_03.max():,}, "
    f"pairs with quantity <= 5={int(matched_quantities_03.le(5).sum()):,}"
)

Cancellation lines considered: 8,905
Cancellation lines matched: 2,895
Cancellation lines unmatched: 6,010
Positive lines excluded: 2,895
Units in positive lines excluded: 225,838
10 largest matched pairs:


,InvoiceNo,C-InvoiceNo,StockCode,Quantity,CustomerID
0,581483,C581484,23843,80995,16446.0
1,541431,C541433,23166,74215,12346.0
2,540815,C550456,21108,3114,15749.0
3,540815,C550456,21175,2000,15749.0
4,540815,C550456,85123A,1930,15749.0
5,556484,C556522,22920,1515,16938.0
6,570094,C570556,20971,1296,16029.0
7,570097,C570556,20971,1296,16029.0
8,569214,C569552,15034,1200,14533.0
9,570097,C570556,22147,840,16029.0


Matched quantity distribution: min=1, median=6, 90th percentile=40, max=80,995, pairs with quantity <= 5=1,362


## Clean population and selected code checks

The table separates raw row counts from rows eligible under the unchanged positive-quantity, positive-price, and product-code rules, before exact-match exclusions.

In [3]:
eligible_before_matching_03 = raw_03.loc[
    ~raw_03["InvoiceNo"].astype(str).str.startswith("C")
    & raw_03["Quantity"].gt(0)
    & raw_03["UnitPrice"].gt(0)
    & raw_03["StockCode"].str.match(PRODUCT_CODE)
]
eligible_matched_03 = eligible_before_matching_03.loc[
    eligible_before_matching_03.index.isin(matched_positive_indices_03)
]
selected_codes_03 = ["23843", "23166", "84347"]
code_rows_03 = []
for stock_code_03 in selected_codes_03:
    raw_code_rows_03 = raw_03.loc[raw_03["StockCode"].eq(stock_code_03)]
    eligible_code_rows_03 = eligible_before_matching_03.loc[
        eligible_before_matching_03["StockCode"].eq(stock_code_03)
    ]
    matched_code_rows_03 = raw_03.loc[
        raw_03.index.isin(matched_positive_indices_03)
        & raw_03["StockCode"].eq(stock_code_03)
    ]
    eligible_matched_code_rows_03 = eligible_matched_03.loc[
        eligible_matched_03["StockCode"].eq(stock_code_03)
    ]
    clean_code_rows_03 = clean_03.loc[clean_03["StockCode"].eq(stock_code_03)]
    code_rows_03.append({
        "StockCode": stock_code_03,
        "Raw rows": len(raw_code_rows_03),
        "Eligible before matching": len(eligible_code_rows_03),
        "Matched positive lines": len(matched_code_rows_03),
        "Matched units": int(matched_code_rows_03["Quantity"].sum()),
        "Eligible matched lines": len(eligible_matched_code_rows_03),
        "Eligible matched units": int(eligible_matched_code_rows_03["Quantity"].sum()),
        "Clean rows": len(clean_code_rows_03),
        "Clean units": int(clean_code_rows_03["Quantity"].sum()),
    })
code_summary_03 = pd.DataFrame(code_rows_03)
display(code_summary_03)

code_23843_match_03 = raw_03.loc[raw_03["StockCode"].eq("23843") & raw_03.index.isin(matched_positive_indices_03)]
code_23166_match_03 = eligible_matched_03.loc[eligible_matched_03["StockCode"].eq("23166")]
code_84347_eligible_03 = eligible_before_matching_03.loc[eligible_before_matching_03["StockCode"].eq("84347")]
code_84347_clean_03 = clean_03.loc[clean_03["StockCode"].eq("84347")]
assert clean_03["StockCode"].eq("23843").sum() == 0
assert int(code_23166_match_03["Quantity"].eq(74215).sum()) == 1
print("Exact eligible matched pairs for 23166:")
display(matched_pairs_03.loc[matched_pairs_03["StockCode"].eq("23166")].reset_index(drop=True))
assert len(code_84347_eligible_03) == len(code_84347_clean_03)
assert code_84347_eligible_03["Quantity"].sum() == code_84347_clean_03["Quantity"].sum()
print(
    "Confirmed: 23843 has 0 clean rows; 84347's eligible rows are fully kept. "
    f"For 23166, {len(code_23166_match_03)} eligible lines match exactly, "
    "including the 74,215-unit line."
)

old_clean_rows_03, old_clean_products_03, old_clean_units_03 = 527003, 3897, 5408492
new_clean_rows_03 = len(clean_03)
new_clean_products_03 = clean_03["StockCode"].nunique()
new_clean_units_03 = int(clean_03["Quantity"].sum())
print(f"Clean rows: {new_clean_rows_03:,} (difference from old: {new_clean_rows_03 - old_clean_rows_03:+,})")
print(f"Products: {new_clean_products_03:,} (difference from old: {new_clean_products_03 - old_clean_products_03:+,})")
print(f"Total units: {new_clean_units_03:,} (difference from old: {new_clean_units_03 - old_clean_units_03:+,})")

,StockCode,Raw rows,Eligible before matching,Matched positive lines,Matched units,Eligible matched lines,Eligible matched units,Clean rows,Clean units
0,23843,2,1,1,80995,1,80995,0,0
1,23166,260,250,5,74480,5,74480,245,3553
2,84347,475,471,0,0,0,0,471,9476


Exact eligible matched pairs for 23166:


,InvoiceNo,C-InvoiceNo,StockCode,Quantity,CustomerID
0,541431,C541433,23166,74215,12346.0
1,553607,C557508,23166,240,16684.0
2,560491,C560540,23166,1,12415.0
3,562374,C562375,23166,12,14911.0
4,570467,C570867,23166,12,12607.0


Confirmed: 23843 has 0 clean rows; 84347's eligible rows are fully kept. For 23166, 5 eligible lines match exactly, including the 74,215-unit line.
Clean rows: 524,894 (difference from old: -2,109)
Products: 3,895 (difference from old: -2)
Total units: 5,351,288 (difference from old: -57,204)


## Unit-price variation diagnostic only

Counts and shares below describe the clean rows only. No interpretation is made here.

In [4]:
price_diagnostic_rows_03 = []
for population_name_03, price_rows_03 in [
    ("All clean rows", clean_03),
    ("CustomerID present", clean_03.loc[clean_03["CustomerID"].notna()]),
    ("CustomerID missing", clean_03.loc[clean_03["CustomerID"].isna()]),
]:
    distinct_prices_by_product_03 = price_rows_03.groupby("StockCode")["UnitPrice"].nunique()
    multiple_price_count_03 = int(distinct_prices_by_product_03.gt(1).sum())
    product_count_03 = len(distinct_prices_by_product_03)
    price_diagnostic_rows_03.append({
        "Population": population_name_03,
        "Products": product_count_03,
        "Products with >1 distinct UnitPrice": multiple_price_count_03,
        "Share": multiple_price_count_03 / product_count_03 if product_count_03 else 0,
    })
price_diagnostics_03 = pd.DataFrame(price_diagnostic_rows_03)
display(price_diagnostics_03)

,Population,Products,Products with >1 distinct UnitPrice,Share
0,All clean rows,3895,3349,0.859820
1,CustomerID present,3645,2618,0.718244
2,CustomerID missing,3388,2730,0.805785


## Matched pairs: time gap, size, full-invoice cancellation

In [5]:
matched_pairs_03b = pd.DataFrame({
    "positive_index": matched_positive_indices_03,
    "cancellation_index": matched_cancellation_indices_03,
})
matched_pairs_03b["InvoiceNo"] = raw_03.loc[matched_pairs_03b["positive_index"], "InvoiceNo"].to_numpy()
matched_pairs_03b["C-InvoiceNo"] = raw_03.loc[matched_pairs_03b["cancellation_index"], "InvoiceNo"].to_numpy()
matched_pairs_03b["StockCode"] = raw_03.loc[matched_pairs_03b["positive_index"], "StockCode"].to_numpy()
matched_pairs_03b["Quantity"] = raw_03.loc[matched_pairs_03b["positive_index"], "Quantity"].to_numpy()
matched_pairs_03b["CustomerID"] = raw_03.loc[matched_pairs_03b["positive_index"], "CustomerID"].to_numpy()
matched_pairs_03b["positive_date"] = raw_03.loc[matched_pairs_03b["positive_index"], "InvoiceDate"].to_numpy()
matched_pairs_03b["cancellation_date"] = raw_03.loc[matched_pairs_03b["cancellation_index"], "InvoiceDate"].to_numpy()
matched_pairs_03b["gap"] = matched_pairs_03b["cancellation_date"] - matched_pairs_03b["positive_date"]
matched_pairs_03b["gap_days"] = matched_pairs_03b["gap"].dt.total_seconds() / 86400

print("Gap percentiles (days):")
display(matched_pairs_03b["gap_days"].quantile([0, 0.25, 0.5, 0.75, 0.9, 1]).rename(index={0.0: "min", 0.25: "25th percentile", 0.5: "median", 0.75: "75th percentile", 0.9: "90th percentile", 1.0: "max"}).to_frame("days"))

matched_pairs_03b["gap_bucket"] = pd.Series(pd.NA, index=matched_pairs_03b.index, dtype="string")
matched_pairs_03b.loc[matched_pairs_03b["gap"].lt(pd.Timedelta(hours=1)), "gap_bucket"] = "<1 hour"
matched_pairs_03b.loc[matched_pairs_03b["gap"].ge(pd.Timedelta(hours=1)) & matched_pairs_03b["gap"].lt(pd.Timedelta(days=1)), "gap_bucket"] = "same day (>=1 hour)"
matched_pairs_03b.loc[matched_pairs_03b["gap"].ge(pd.Timedelta(days=1)) & matched_pairs_03b["gap"].le(pd.Timedelta(days=7)), "gap_bucket"] = "1-7 days"
matched_pairs_03b.loc[matched_pairs_03b["gap"].gt(pd.Timedelta(days=7)) & matched_pairs_03b["gap"].le(pd.Timedelta(days=30)), "gap_bucket"] = "8-30 days"
matched_pairs_03b.loc[matched_pairs_03b["gap"].gt(pd.Timedelta(days=30)), "gap_bucket"] = ">30 days"
matched_pairs_03b["quantity_bucket"] = pd.Series(pd.NA, index=matched_pairs_03b.index, dtype="string")
matched_pairs_03b.loc[matched_pairs_03b["Quantity"].le(5), "quantity_bucket"] = "<=5"
matched_pairs_03b.loc[matched_pairs_03b["Quantity"].between(6, 50), "quantity_bucket"] = "6-50"
matched_pairs_03b.loc[matched_pairs_03b["Quantity"].between(51, 999), "quantity_bucket"] = "51-999"
matched_pairs_03b.loc[matched_pairs_03b["Quantity"].ge(1000), "quantity_bucket"] = ">=1000"
gap_bucket_order_03b = ["<1 hour", "same day (>=1 hour)", "1-7 days", "8-30 days", ">30 days"]
quantity_bucket_order_03b = ["<=5", "6-50", "51-999", ">=1000"]

print("Pair counts and units by gap bucket:")
display(matched_pairs_03b.groupby("gap_bucket", observed=False).agg(pairs=("Quantity", "size"), units=("Quantity", "sum")).reindex(gap_bucket_order_03b, fill_value=0))
print("Pair counts and units by quantity bucket:")
display(matched_pairs_03b.groupby("quantity_bucket", observed=False).agg(pairs=("Quantity", "size"), units=("Quantity", "sum")).reindex(quantity_bucket_order_03b, fill_value=0))

print("Pair-count crosstab: gap bucket x quantity bucket")
matched_pair_counts_crosstab_03b = matched_pairs_03b.groupby(["gap_bucket", "quantity_bucket"], observed=False).size().unstack(fill_value=0).reindex(index=gap_bucket_order_03b, columns=quantity_bucket_order_03b, fill_value=0)
display(matched_pair_counts_crosstab_03b)
print("Unit-sum crosstab: gap bucket x quantity bucket")
matched_pair_units_crosstab_03b = matched_pairs_03b.groupby(["gap_bucket", "quantity_bucket"], observed=False)["Quantity"].sum().unstack(fill_value=0).reindex(index=gap_bucket_order_03b, columns=quantity_bucket_order_03b, fill_value=0)
display(matched_pair_units_crosstab_03b)

matched_pairs_selected_03b = matched_pairs_03b.loc[matched_pairs_03b["Quantity"].ge(1000) | matched_pairs_03b["gap"].lt(pd.Timedelta(hours=1))].copy()
matched_pairs_selected_03b["gap_hours"] = matched_pairs_selected_03b["gap"].dt.total_seconds() / 3600
matched_pairs_selected_03b = matched_pairs_selected_03b.sort_values(["Quantity", "gap"], ascending=[False, True], kind="mergesort")
print(f"Pairs meeting Quantity >= 1000 or gap < 1 hour: {len(matched_pairs_selected_03b)}; shown: {min(40, len(matched_pairs_selected_03b))}")
display(matched_pairs_selected_03b[["InvoiceNo", "C-InvoiceNo", "StockCode", "Quantity", "CustomerID", "positive_date", "cancellation_date", "gap", "gap_hours"]].head(40).reset_index(drop=True))

matched_positive_index_set_03b = set(matched_positive_indices_03)
matched_invoice_rows_03b = []
for matched_invoice_03b in matched_pairs_03b["InvoiceNo"].drop_duplicates():
    invoice_raw_rows_03b = raw_03.loc[raw_03["InvoiceNo"].eq(matched_invoice_03b)]
    invoice_matched_rows_03b = invoice_raw_rows_03b.loc[invoice_raw_rows_03b.index.isin(matched_positive_index_set_03b)]
    matched_invoice_rows_03b.append({
        "InvoiceNo": matched_invoice_03b,
        "raw_lines": len(invoice_raw_rows_03b),
        "matched_lines": len(invoice_matched_rows_03b),
        "matched_units": int(invoice_matched_rows_03b["Quantity"].sum()),
        "fully_cancelled": len(invoice_raw_rows_03b) == len(invoice_matched_rows_03b),
    })
matched_invoice_detail_03b = pd.DataFrame(matched_invoice_rows_03b)
matched_invoice_detail_03b["invoice_status"] = matched_invoice_detail_03b["fully_cancelled"].map({True: "fully cancelled", False: "partly cancelled"})
print("Matched positive invoices: full vs partial cancellation:")
display(matched_invoice_detail_03b.groupby("invoice_status", sort=False).agg(invoices=("InvoiceNo", "size"), raw_lines=("raw_lines", "sum"), matched_lines=("matched_lines", "sum"), units=("matched_units", "sum")).reindex(["fully cancelled", "partly cancelled"], fill_value=0))

print("Five matched pairs for StockCode 23166 with gaps:")
display(matched_pairs_03b.loc[matched_pairs_03b["StockCode"].eq("23166"), ["InvoiceNo", "C-InvoiceNo", "StockCode", "Quantity", "CustomerID", "positive_date", "cancellation_date", "gap", "gap_days"]].sort_values(["positive_date", "InvoiceNo"], kind="mergesort").reset_index(drop=True))

Gap percentiles (days):


,days
min,0.000694
25th percentile,1.806944
median,9.107639
75th percentile,28.995486
90th percentile,70.095833
max,368.187500


Pair counts and units by gap bucket:


,pairs,units
gap_bucket,,
<1 hour,464,172637
same day (>=1 hour),213,7180
1-7 days,605,15415
8-30 days,904,11528
>30 days,709,19078


Pair counts and units by quantity bucket:


...

,InvoiceNo,C-InvoiceNo,StockCode,Quantity,CustomerID,positive_date,cancellation_date,gap,gap_hours
0,581483,C581484,23843,80995,16446.0,2011-12-09 09:15:00,2011-12-09 09:27:00,0 days 00:12:00,0.200000
1,541431,C541433,23166,74215,12346.0,2011-01-18 10:01:00,2011-01-18 10:17:00,0 days 00:16:00,0.266667
2,540815,C550456,21108,3114,15749.0,2011-01-11 12:55:00,2011-04-18 13:08:00,97 days 00:13:00,2328.216667
3,540815,C550456,21175,2000,15749.0,2011-01-11 12:55:00,2011-04-18 13:08:00,97 days 00:13:00,2328.216667
4,540815,C550456,85123A,1930,15749.0,2011-01-11 12:55:00,2011-04-18 13:08:00,97 days 00:13:00,2328.216667
5,556484,C556522,22920,1515,16938.0,2011-06-12 13:17:00,2011-06-13 11:21:00,0 days 22:04:00,22.066667
6,570097,C570556,20971,1296,16029.0,2011-10-07 12:05:00,2011-10-11 11:10:00,3 days 23:05:00,95.083333
7,570094,C570556,20971,1296,16029.0,2011-10-07 11:56:00,2011-10-11 11:10:00,3 days 23:14:00,95.233333
8,569214,C569552,15034,1200,14533.0,2011-10-02 12:22:00,2011-10-04 17:44:00,2 days 05:22:00,53.366667
9,572340,C572343,22102,720,14607.0,2011-10-24 10:36:00,2011-10-24 10:40:00,0 days 00:04:00,0.066667


Matched positive invoices: full vs partial cancellation:


,invoices,raw_lines,matched_lines,units
invoice_status,,,,
fully cancelled,144,975,975,198269
partly cancelled,1226,41750,1920,27569


Five matched pairs for StockCode 23166 with gaps:


,InvoiceNo,C-InvoiceNo,StockCode,Quantity,CustomerID,positive_date,cancellation_date,gap,gap_days
0,541431,C541433,23166,74215,12346.0,2011-01-18 10:01:00,2011-01-18 10:17:00,0 days 00:16:00,0.011111
1,553607,C557508,23166,240,16684.0,2011-05-18 10:47:00,2011-06-20 16:13:00,33 days 05:26:00,33.226389
2,560491,C560540,23166,1,12415.0,2011-07-19 10:51:00,2011-07-19 12:26:00,0 days 01:35:00,0.065972
3,562374,C562375,23166,12,14911.0,2011-08-04 14:40:00,2011-08-04 14:46:00,0 days 00:06:00,0.004167
4,570467,C570867,23166,12,12607.0,2011-10-10 16:06:00,2011-10-12 16:17:00,2 days 00:11:00,2.007639
